# Swiss Legal Retrieval — Phase 1: BM25 Baseline

**Competition:** [LLM Agentic Legal Information Retrieval](https://www.kaggle.com/competitions/llm-agentic-legal-information-retrieval)

**Strategy:** BM25 keyword retrieval over `laws_de.csv`.  
No GPU, no embeddings — gets us on the leaderboard fast.

**Metric:** Macro F1 (citation-level, averaged across queries)

---
**Roadmap:**
- Phase 1 (this notebook): BM25 baseline
- Phase 2: Multilingual embeddings + query translation
- Phase 3: Hybrid BM25 + embeddings + cross-encoder reranker
- Phase 4: Agentic retrieval with small local LLM

In [ ]:
# Install BM25 library (allowed in offline Kaggle notebooks)
!pip install rank_bm25 -q

In [ ]:
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
import re
import os

print('Libraries loaded ✅')

## ⚙️ Config

In [ ]:
DATA_DIR    = '/kaggle/input/llm-agentic-legal-information-retrieval'
OUTPUT_PATH = '/kaggle/working/submission.csv'

# How many citations to return per query (precision/recall tradeoff)
TOP_K = 5

# Set True to also search court decisions (2.4GB file — slower but more complete)
USE_COURT = False

## 📂 Load Data

In [ ]:
print('Loading test queries...')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'  → {len(test_df)} test queries')
test_df.head(3)

In [ ]:
print('Loading laws corpus...')
laws_df = pd.read_csv(f'{DATA_DIR}/laws_de.csv')
print(f'  → {len(laws_df):,} law snippets')
laws_df.head(3)

In [ ]:
if USE_COURT:
    print('Loading court considerations (2.4GB — this takes a while)...')
    court_df = pd.read_csv(f'{DATA_DIR}/court_considerations.csv')
    print(f'  → {len(court_df):,} court decisions')
    corpus_df = pd.concat([laws_df, court_df], ignore_index=True)
else:
    corpus_df = laws_df

print(f'\nTotal corpus: {len(corpus_df):,} documents')

## 🔤 Tokenizer

Simple lowercase + regex tokenizer.  
Keeps German umlauts (ä,ö,ü,ß) and dots/slashes (needed for citation IDs like `BGE 139 I 2 E. 3.1`).  

**Phase 2 improvement:** translate English queries to German first, so keyword overlap improves dramatically.

In [ ]:
def simple_tokenize(text: str) -> list:
    if not isinstance(text, str):
        return []
    text = text.lower()
    tokens = re.findall(r'[a-z0-9äöüß./]+', text)
    return tokens

# Quick sanity check
print(simple_tokenize('What are the inheritance rights of a surviving spouse under Art. 462 ZGB?'))

## 🏗️ Build BM25 Index

We concatenate `citation + text` so citation keywords (e.g., `ZGB`, `OR`, `BGE`) also influence matching scores.

In [ ]:
print('Building BM25 index...')
corpus_texts    = corpus_df['citation'].fillna('') + ' ' + corpus_df['text'].fillna('')
tokenized_corpus = [simple_tokenize(doc) for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)
print('BM25 index ready! ✅')

## 🔍 Retrieve Citations

In [ ]:
print(f'Retrieving top-{TOP_K} citations per query...')
predictions = []

for _, row in test_df.iterrows():
    query_id = row['query_id']
    query    = row['query']
    
    query_tokens = simple_tokenize(query)
    
    if not query_tokens:
        predictions.append({'query_id': query_id, 'predicted_citations': ''})
        continue
    
    # Score all documents
    scores = bm25.get_scores(query_tokens)
    
    # Take top-K, filter zero-score docs
    top_k_idx = np.argsort(scores)[::-1][:TOP_K]
    top_k_idx = [i for i in top_k_idx if scores[i] > 0]
    
    top_citations = corpus_df.iloc[top_k_idx]['citation'].tolist()
    
    predictions.append({
        'query_id': query_id,
        'predicted_citations': ';'.join(top_citations)
    })

print(f'Done! {len(predictions)} predictions made.')

## 💾 Write Submission File

In [ ]:
submission_df = pd.DataFrame(predictions)
submission_df.to_csv(OUTPUT_PATH, index=False)
print(f'Submission saved to {OUTPUT_PATH} ✅')
submission_df.head(10)

## 📊 Local Validation (val.csv)

Gives an estimate of Macro F1 before submitting.

In [ ]:
val_path = f'{DATA_DIR}/val.csv'
if os.path.exists(val_path):
    val_df = pd.read_csv(val_path)
    
    val_preds = []
    for _, row in val_df.iterrows():
        tokens  = simple_tokenize(row['query'])
        if not tokens:
            val_preds.append(set())
            continue
        scores  = bm25.get_scores(tokens)
        top_idx = np.argsort(scores)[::-1][:TOP_K]
        top_idx = [i for i in top_idx if scores[i] > 0]
        val_preds.append(set(corpus_df.iloc[top_idx]['citation'].tolist()))
    
    f1_scores = []
    for i, row in val_df.iterrows():
        gold = set(str(row['gold_citations']).split(';'))
        pred = val_preds[i]
        if not gold and not pred:
            f1_scores.append(1.0)
        elif not gold or not pred:
            f1_scores.append(0.0)
        else:
            tp = len(gold & pred)
            p  = tp / len(pred) if pred else 0.0
            r  = tp / len(gold) if gold else 0.0
            f1_scores.append(2*p*r / (p+r) if (p+r) > 0 else 0.0)
    
    print(f'Val Macro F1: {np.mean(f1_scores):.4f}')
    print(f'Per-query:    {[round(s,3) for s in f1_scores]}')
else:
    print('val.csv not found — skipping local evaluation')